In [30]:
# importando as bibliotecas
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [26]:
# dados
x = [[22, 60], [30, 80], [18, 50], [35, 90], [25, 70], [28, 75]]
y = [[1], [0], [1], [0], [1], [0]]

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [28]:
# dataset
class ClimaDataset(Dataset):
    def __init__(self, x, y):
        t = torch.tensor(x).float()
        self.x = (t - t.min(0).values) / (t.max(0).values - t.min(0).values)
        self.y = torch.tensor(y).float()

    def __len__(self):
        return len(self.x)

    def __getitem__(self, ix):
        return self.x[ix], self.y[ix]

ds = ClimaDataset(x, y)
print(len(ds))
print(ds[0])

6
(tensor([0.2353, 0.2500]), tensor([1.]))


In [31]:
# dataloader
dl = DataLoader(ds, batch_size=3, shuffle=True)
for xb, yb in dl:
    print("x:", xb.tolist(), " y:", yb.tolist())

x: [[1.0, 1.0], [0.5882353186607361, 0.625], [0.7058823704719543, 0.75]]  y: [[0.0], [0.0], [0.0]]
x: [[0.0, 0.0], [0.4117647111415863, 0.5], [0.23529411852359772, 0.25]]  y: [[1.0], [1.0], [1.0]]


In [32]:
# rede simples pra classificação binaria
class RedeClima(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 8)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(8, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.activation(x)
        return self.layer2(x)

model = RedeClima().to(device)
loss_fn = nn.BCEWithLogitsLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)

In [33]:
# loop de treino
losses = []
for _ in range(100):
    for xb, yb in dl:
        opt.zero_grad()
        loss_value = loss_fn(model(xb), yb)
        loss_value.backward()
        opt.step()
        losses.append(loss_value.item())

In [35]:
print(f"erro inicial:{losses[0]:.4f}")
print(f"erro final:{losses[-1]:.4f}")

erro inicial:0.6693
erro final:0.4950
